# Bootstrap Uncertainty Intervals

Demonstrates semiparametric and residual bootstrap for parameter uncertainty.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pystmomo as ps

data = ps.load_ew_male()
fit = ps.lc().fit(data.deaths, data.exposures,
                  ages=data.ages, years=data.years)

## Semiparametric Bootstrap

In [ ]:
boot_sp = ps.semiparametric_bootstrap(fit, nboot=200, seed=0)
print(boot_sp)

## Residual Bootstrap

In [ ]:
boot_res = ps.residual_bootstrap(fit, nboot=200, seed=1)
print(boot_res)

## Compare Bootstrap Distributions of κ_t

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, boot, title in zip(axes,
                            [boot_sp, boot_res],
                            ['Semiparametric', 'Residual']):
    lo, hi = boot.parameter_ci('kt', level=0.95)
    kt_b = boot.kt_b  # shape (nboot, 1, n_years)
    
    # Plot each bootstrap replicate lightly
    for b in range(len(boot.fits)):
        ax.plot(data.years, kt_b[b, 0, :], color='steelblue', alpha=0.05, lw=0.5)
    
    ax.plot(data.years, fit.kt[0], 'k-', lw=2, label='Fitted')
    ax.fill_between(data.years, lo[0], hi[0], alpha=0.3, color='steelblue', label='95% CI')
    ax.set_xlabel('Year'); ax.set_ylabel('κ_t')
    ax.set_title(f'{title} Bootstrap (n=200)')
    ax.legend()

plt.tight_layout()
plt.show()

## Bootstrap Distribution of β_x

In [ ]:
lo_bx, hi_bx = boot_sp.parameter_ci('bx', level=0.95)

plt.figure(figsize=(8, 4))
plt.plot(data.ages, fit.bx[:, 0], 'k-', lw=2, label='Fitted β_x')
plt.fill_between(data.ages, lo_bx[:, 0], hi_bx[:, 0],
                 alpha=0.3, color='steelblue', label='95% CI')
plt.xlabel('Age'); plt.ylabel('β_x')
plt.title('Age Sensitivity — Bootstrap CI')
plt.legend(); plt.tight_layout(); plt.show()

## Deviance Distribution Across Bootstrap Replicates

In [ ]:
deviances = [f.deviance for f in boot_sp.fits]
plt.figure(figsize=(8, 4))
plt.hist(deviances, bins=30, color='steelblue', edgecolor='white')
plt.axvline(fit.deviance, color='red', lw=2, label=f'Base deviance={fit.deviance:.1f}')
plt.xlabel('Deviance'); plt.ylabel('Count')
plt.title('Bootstrap Deviance Distribution')
plt.legend(); plt.tight_layout(); plt.show()